In [20]:
# =========================================================
# Klinik Parametre Hesabı - Temel Fonksiyonlar
# =========================================================

import numpy as np


# ---------------------------------------------------------
# 1) Voxel hacmini hesapla
# ---------------------------------------------------------
# spacing = (dx, dy, dz) olacak
# birimi mm cinsinden bekliyoruz
#
# Örnek:
# dx = 1.37 mm
# dy = 1.37 mm
# dz = 10.0 mm
#
# Bir voxelin fiziksel hacmi:
# mm^3 cinsinden = dx * dy * dz
#
# mL'ye çevirmek için 1000'e bölüyoruz
# çünkü 1 mL = 1000 mm^3
# ---------------------------------------------------------
def compute_voxel_volume_ml(spacing):
    dx, dy, dz = spacing
    voxel_volume_ml = (dx * dy * dz) / 1000.0
    return voxel_volume_ml


# ---------------------------------------------------------
# 2) Belirli bir sınıfın hacmini hesapla
# ---------------------------------------------------------
# mask      : (D, H, W) biçiminde sınıf maskesi
# class_id  : hangi sınıfın hacmini hesaplayacağımız
# voxel_volume_ml : tek voxelin mL cinsinden hacmi
#
# Mantık:
# - maskede class_id'ye eşit olan voxel sayısını bul
# - bunu tek voxel hacmi ile çarp
# ---------------------------------------------------------
def compute_class_volume_ml(mask, class_id, voxel_volume_ml):
    voxel_count = int(np.sum(mask == class_id))
    volume_ml = voxel_count * voxel_volume_ml
    return volume_ml


# ---------------------------------------------------------
# 3) LV üzerinden EDV, ESV ve EF hesapla
# ---------------------------------------------------------
# ed_mask : end-diastole (ED) fazına ait segmentasyon maskesi
# es_mask : end-systole (ES) fazına ait segmentasyon maskesi
# spacing : (dx, dy, dz)
# lv_class_id : LV cavity sınıf etiketi
#
# Bu projede sınıf sırası:
# 0 = BG
# 1 = RV
# 2 = MYO
# 3 = LV
#
# Bu yüzden varsayılan lv_class_id = 3
# ---------------------------------------------------------
def compute_lv_functional_metrics(ed_mask, es_mask, spacing, lv_class_id=3):
    # Önce bir voxelin mL cinsinden hacmini bul
    voxel_volume_ml = compute_voxel_volume_ml(spacing)

    # ED fazındaki LV hacmi = EDV
    edv_ml = compute_class_volume_ml(ed_mask, lv_class_id, voxel_volume_ml)

    # ES fazındaki LV hacmi = ESV
    esv_ml = compute_class_volume_ml(es_mask, lv_class_id, voxel_volume_ml)

    # EF formülü:
    # EF = (EDV - ESV) / EDV * 100
    #
    # Ama EDV sıfırsa bölme hatası olmaması için kontrol koyuyoruz
    if edv_ml > 0:
        ef_percent = ((edv_ml - esv_ml) / edv_ml) * 100.0
    else:
        ef_percent = None

    # Sonucu sözlük olarak döndürmek ileride JSON'a çevirmeyi kolaylaştırır
    results = {
        "voxel_volume_ml": voxel_volume_ml,
        "lv_edv_ml": edv_ml,
        "lv_esv_ml": esv_ml,
        "lv_ef_percent": ef_percent,
    }

    return results


# ---------------------------------------------------------
# 4) Basit kalite kontrolleri
# ---------------------------------------------------------
# Amaç:
# - saçma değerleri erken yakalamak
# - ileride VQA katmanına daha güvenilir veri vermek
# ---------------------------------------------------------
def run_quality_checks(metrics):
    edv = metrics["lv_edv_ml"]
    esv = metrics["lv_esv_ml"]
    ef = metrics["lv_ef_percent"]

    checks = {
        "lv_present_in_ed": edv > 0,
        "lv_present_in_es": esv > 0,
        "edv_greater_than_esv": edv >= esv,
        "ef_in_valid_range": (ef is not None) and (0 <= ef <= 100),
    }

    warnings = []

    if not checks["lv_present_in_ed"]:
        warnings.append("ED fazında LV segmentasyonu bulunamadı.")

    if not checks["lv_present_in_es"]:
        warnings.append("ES fazında LV segmentasyonu bulunamadı.")

    if not checks["edv_greater_than_esv"]:
        warnings.append(
            "EDV, ESV'den küçük çıktı. Faz sırası veya segmentasyon kontrol edilmeli."
        )

    if not checks["ef_in_valid_range"]:
        warnings.append("EF beklenen aralık dışında.")

    return checks, warnings

In [21]:
# =========================================================
# Klinik Parametre Hesabı - Basit Test
# =========================================================

# Örnek spacing (mm)
spacing = (1.37, 1.37, 10.0)

# Örnek ED ve ES maskeleri oluşturalım
# Burada küçük 3D maskeler kullanıyoruz
# class 3 = LV

ed_mask = np.zeros((4, 8, 8), dtype=np.uint8)
es_mask = np.zeros((4, 8, 8), dtype=np.uint8)

# ED fazında LV daha büyük olsun
ed_mask[:, 2:6, 2:6] = 3  # 4x4 alan, 4 slice boyunca

# ES fazında LV daha küçük olsun
es_mask[:, 3:5, 3:5] = 3  # 2x2 alan, 4 slice boyunca

metrics = compute_lv_functional_metrics(
    ed_mask=ed_mask, es_mask=es_mask, spacing=spacing, lv_class_id=3
)

checks, warnings = run_quality_checks(metrics)

print("=== KLINIK PARAMETRELER ===")
for k, v in metrics.items():
    print(f"{k}: {v}")

print("\n=== KALITE KONTROLLERI ===")
for k, v in checks.items():
    print(f"{k}: {v}")

print("\n=== UYARILAR ===")
print(warnings if warnings else "Uyarı yok.")

=== KLINIK PARAMETRELER ===
voxel_volume_ml: 0.018769
lv_edv_ml: 1.201216
lv_esv_ml: 0.300304
lv_ef_percent: 75.0

=== KALITE KONTROLLERI ===
lv_present_in_ed: True
lv_present_in_es: True
edv_greater_than_esv: True
ef_in_valid_range: True

=== UYARILAR ===
Uyarı yok.


In [22]:
# =========================================================
# Adım 3.1 - NIfTI maske yükleme yardımcı fonksiyonları
# =========================================================

import os
import nibabel as nib
import numpy as np


# ---------------------------------------------------------
# NIfTI dosyasını yükle
# ---------------------------------------------------------
# Çıktı:
# - volume: (D, H, W) olacak şekilde numpy array
# - spacing: (dx, dy, dz)
#
# Not:
# nibabel çoğu zaman veriyi (H, W, D) olarak okur.
# Biz klinik pipeline boyunca (D, H, W) ile ilerlemek istediğimiz için
# eksen sırasını değiştiriyoruz.
# ---------------------------------------------------------
def load_nifti_mask_as_dhw(nifti_path):
    nii = nib.load(nifti_path)

    # Veri float gelebilir; maske olduğu için integer'a çevireceğiz
    vol = nii.get_fdata()

    # spacing bilgisi
    # header zooms çoğu zaman (dx, dy, dz) verir
    spacing = nii.header.get_zooms()[:3]

    # (H, W, D) -> (D, H, W)
    vol_dhw = np.transpose(vol, (2, 0, 1))

    # Maske olduğundan integer etikete çevir
    vol_dhw = np.rint(vol_dhw).astype(np.uint8)

    return vol_dhw, spacing


# ---------------------------------------------------------
# Basit bilgi yazdırma fonksiyonu
# ---------------------------------------------------------
def print_mask_info(name, mask, spacing):
    print(f"{name} shape (D,H,W): {mask.shape}")
    print(f"{name} spacing (dx,dy,dz): {spacing}")
    print(f"{name} unique labels: {np.unique(mask)}")

In [23]:
# =========================================================
# Adım 3.2 - Gerçek ED/ES GT maskelerini yükle
# =========================================================

# Örnek hasta
patient_id = "patient101"

# Notebook'un notebooks/ klasöründe olduğunu varsayıyoruz
demo_root = os.path.join("..", "backend", "data", "demo", patient_id)

# Bu hasta için ED ve ES GT dosyaları
ed_gt_path = os.path.join(demo_root, "patient101_frame01_gt.nii")
es_gt_path = os.path.join(demo_root, "patient101_frame14_gt.nii")

print("ED GT path:", ed_gt_path)
print("ES GT path:", es_gt_path)
print("ED exists:", os.path.exists(ed_gt_path))
print("ES exists:", os.path.exists(es_gt_path))

ed_mask_gt, ed_spacing = load_nifti_mask_as_dhw(ed_gt_path)
es_mask_gt, es_spacing = load_nifti_mask_as_dhw(es_gt_path)

print_mask_info("ED GT", ed_mask_gt, ed_spacing)
print()
print_mask_info("ES GT", es_mask_gt, es_spacing)

ED GT path: ..\backend\data\demo\patient101\patient101_frame01_gt.nii
ES GT path: ..\backend\data\demo\patient101\patient101_frame14_gt.nii
ED exists: True
ES exists: True
ED GT shape (D,H,W): (10, 232, 256)
ED GT spacing (dx,dy,dz): (np.float32(1.64062), np.float32(1.64062), np.float32(10.0))
ED GT unique labels: [0 1 2 3]

ES GT shape (D,H,W): (10, 232, 256)
ES GT spacing (dx,dy,dz): (np.float32(1.64062), np.float32(1.64062), np.float32(10.0))
ES GT unique labels: [0 1 2 3]


In [24]:
# =========================================================
# Adım 3.3 - Spacing ve shape kontrolü
# =========================================================

print("ED spacing:", ed_spacing)
print("ES spacing:", es_spacing)

print("ED shape:", ed_mask_gt.shape)
print("ES shape:", es_mask_gt.shape)

same_spacing = np.allclose(ed_spacing, es_spacing)
same_shape = ed_mask_gt.shape == es_mask_gt.shape

print("Spacing aynı mı?:", same_spacing)
print("Shape aynı mı?:", same_shape)

ED spacing: (np.float32(1.64062), np.float32(1.64062), np.float32(10.0))
ES spacing: (np.float32(1.64062), np.float32(1.64062), np.float32(10.0))
ED shape: (10, 232, 256)
ES shape: (10, 232, 256)
Spacing aynı mı?: True
Shape aynı mı?: True


In [25]:
# =========================================================
# Adım 3.4 - Gerçek GT maskelerden klinik parametre hesabı
# =========================================================

# ED ve ES spacing aynı kabul ediyoruz
spacing = ed_spacing

metrics_gt = compute_lv_functional_metrics(
    ed_mask=ed_mask_gt,
    es_mask=es_mask_gt,
    spacing=spacing,
    lv_class_id=3,  # bu projede LV etiketi = 3
)

checks_gt, warnings_gt = run_quality_checks(metrics_gt)

print("=== GT UZERINDEN KLINIK PARAMETRELER ===")
for k, v in metrics_gt.items():
    print(f"{k}: {v}")

print("\n=== GT KALITE KONTROLLERI ===")
for k, v in checks_gt.items():
    print(f"{k}: {v}")

print("\n=== GT UYARILAR ===")
print(warnings_gt if warnings_gt else "Uyarı yok.")

=== GT UZERINDEN KLINIK PARAMETRELER ===
voxel_volume_ml: 0.026916339993476868
lv_edv_ml: 222.59812927246094
lv_esv_ml: 156.00711059570312
lv_ef_percent: 29.915353775024414

=== GT KALITE KONTROLLERI ===
lv_present_in_ed: True
lv_present_in_es: True
edv_greater_than_esv: True
ef_in_valid_range: True

=== GT UYARILAR ===
Uyarı yok.


In [26]:
# =========================================================
# Adım 4A - 3D model checkpoint kontrolü
# =========================================================

import os
import torch

# Model dosya yolu
ckpt_path = os.path.join("..", "backend", "models", "best_model.pt")

print("Checkpoint path:", ckpt_path)
print("Dosya var mı?:", os.path.exists(ckpt_path))

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

print("\n=== CHECKPOINT ANAHTARLARI ===")
print(ckpt.keys())

print("\n=== CHECKPOINT ICERIGI ===")
print("epoch:", ckpt.get("epoch", None))
print("best_val_dice:", ckpt.get("best_val_dice", None))
print("run_tag:", ckpt.get("run_tag", None))

Checkpoint path: ..\backend\models\best_model.pt
Dosya var mı?: True

=== CHECKPOINT ANAHTARLARI ===
dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'best_val_dice', 'run_tag', 'run_time'])

=== CHECKPOINT ICERIGI ===
epoch: 49
best_val_dice: 0.8738915048506688
run_tag: unet3d_gn_wce_dice_seed42


In [27]:
# =========================
# 3D U-Net Mimarisi
# =========================

import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------------------------------------------
# Hiperparametreler / temel sabitler
# ---------------------------------------------------
NUM_CLASSES = 4  # background + RV + MYO + LV
BASE_CH = 32  # İlk kanal sayısı; VRAM yeterliyse 32 de denenebilir


# ---------------------------------------------------
# Yardımcı fonksiyon:
# Decoder'dan gelen tensor ile encoder skip tensorünün
# D,H,W boyutları farklıysa eşitlemek için kullanılır.
# ---------------------------------------------------
def match_size_3d(src, ref):
    """
    src tensorünü ref tensorünün uzamsal boyutlarına getirir.

    Girdi:
        src: (B, C, D, H, W)
        ref: (B, C, D, H, W)

    Çıktı:
        src'nin D,H,W boyutları ref ile eşitlenmiş hali

    Neden gerekli?
    --------------
    3D veride depth sabit olmadığı için pooling/upsampling sonrası
    bazen skip connection ile decoder çıktısının boyutları tam tutmayabilir.

    Bu fonksiyon, concat öncesinde güvenli bir eşitleme yapar.
    """
    if src.shape[2:] != ref.shape[2:]:
        src = F.interpolate(
            src,
            size=ref.shape[2:],  # hedef D,H,W boyutu
            mode="trilinear",
            align_corners=False,
        )
    return src


# ---------------------------------------------------
# Temel yapı taşı:
# Conv3d + BN + ReLU + Conv3d + BN + ReLU
# ---------------------------------------------------
class DoubleConv3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        # Batch size küçük olduğu için GroupNorm tercih ediyoruz
        # 32, 64, 128, 256 gibi kanal sayılarında 8 grup uygundur
        num_groups = 8 if out_ch % 8 == 0 else 4 if out_ch % 4 == 0 else 1

        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=num_groups, num_channels=out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=num_groups, num_channels=out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


# ---------------------------------------------------
# 3D U-Net
# ---------------------------------------------------
class UNet3D(nn.Module):
    """
    3D U-Net mimarisi.

    Girdi:
        (B, 1, D, H, W)

    Çıktı:
        (B, NUM_CLASSES, D, H, W)

    Tasarım kararı:
    ---------------
    İlk iki pooling katmanında depth korunur:
        kernel_size=(1,2,2)

    Sebep:
    - Cardiac MRI hacimlerinde depth genelde sınırlıdır
    - Depth'i erken küçültmek anatomik bilgiyi azaltabilir
    - H ve W büyük olduğu için önce in-plane küçültme daha mantıklıdır
    """

    def __init__(self, in_channels=1, num_classes=4, base_ch=16):
        super().__init__()

        # -----------------------------
        # Encoder
        # -----------------------------
        # 1. seviye
        self.enc1 = DoubleConv3D(in_channels, base_ch)

        # İlk pooling:
        # depth korunur, sadece H ve W küçülür
        self.pool1 = nn.MaxPool3d(kernel_size=(1, 2, 2))

        # 2. seviye
        self.enc2 = DoubleConv3D(base_ch, base_ch * 2)

        # İkinci pooling:
        # yine depth korunur
        self.pool2 = nn.MaxPool3d(kernel_size=(1, 2, 2))

        # 3. seviye
        self.enc3 = DoubleConv3D(base_ch * 2, base_ch * 4)

        # Üçüncü pooling:
        # burada artık depth de küçültülebilir
        self.pool3 = nn.MaxPool3d(kernel_size=(2, 2, 2))

        # -----------------------------
        # Bottleneck
        # -----------------------------
        self.bottleneck = DoubleConv3D(base_ch * 4, base_ch * 8)

        # -----------------------------
        # Decoder
        # -----------------------------
        # pool3'ün tersine uygun upsampling
        self.up3 = nn.ConvTranspose3d(
            base_ch * 8, base_ch * 4, kernel_size=(2, 2, 2), stride=(2, 2, 2)
        )
        self.dec3 = DoubleConv3D(base_ch * 8, base_ch * 4)

        # pool2 depth'i koruyordu → burada da depth'i koruyan upsampling
        self.up2 = nn.ConvTranspose3d(
            base_ch * 4, base_ch * 2, kernel_size=(1, 2, 2), stride=(1, 2, 2)
        )
        self.dec2 = DoubleConv3D(base_ch * 4, base_ch * 2)

        # pool1 depth'i koruyordu → burada da depth'i koruyan upsampling
        self.up1 = nn.ConvTranspose3d(
            base_ch * 2, base_ch, kernel_size=(1, 2, 2), stride=(1, 2, 2)
        )
        self.dec1 = DoubleConv3D(base_ch * 2, base_ch)

        # -----------------------------
        # Çıkış katmanı
        # -----------------------------
        # Her voxel için sınıf logits üretir
        self.out_conv = nn.Conv3d(base_ch, num_classes, kernel_size=1)

    def forward(self, x):
        # =================================================
        # ENCODER
        # =================================================

        # 1. seviye özellik çıkarımı
        e1 = self.enc1(x)  # (B, base, D, H, W)
        p1 = self.pool1(e1)  # depth aynı, H/W yarıya iner

        # 2. seviye
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)  # depth aynı, H/W tekrar yarıya iner

        # 3. seviye
        e3 = self.enc3(p2)
        p3 = self.pool3(e3)  # burada depth de küçülür

        # =================================================
        # BOTTLENECK
        # =================================================
        b = self.bottleneck(p3)

        # =================================================
        # DECODER
        # =================================================

        # 3. skip bağlantısı
        u3 = self.up3(b)
        u3 = match_size_3d(u3, e3)  # Boyutlar tam tutmazsa eşitle
        u3 = torch.cat([u3, e3], dim=1)  # Skip connection
        d3 = self.dec3(u3)

        # 2. skip bağlantısı
        u2 = self.up2(d3)
        u2 = match_size_3d(u2, e2)
        u2 = torch.cat([u2, e2], dim=1)
        d2 = self.dec2(u2)

        # 1. skip bağlantısı
        u1 = self.up1(d2)
        u1 = match_size_3d(u1, e1)
        u1 = torch.cat([u1, e1], dim=1)
        d1 = self.dec1(u1)

        # =================================================
        # OUTPUT
        # =================================================
        out = self.out_conv(d1)
        return out


# ---------------------------------------------------
# Model oluşturma yardımcı fonksiyonu
# ---------------------------------------------------
def get_model_3d(model_name="unet3d"):
    """
    İleride farklı 3D mimariler eklemek istersen bu fonksiyon işine yarar.
    Şimdilik sadece temiz baseline olarak UNet3D kullanıyoruz.
    """
    if model_name == "unet3d":
        return UNet3D(in_channels=1, num_classes=NUM_CLASSES, base_ch=BASE_CH)
    else:
        raise ValueError(f"Bilinmeyen model adı: {model_name}")


# ---------------------------------------------------
# Hızlı model testi
# ---------------------------------------------------
model = get_model_3d("unet3d")
print(model.__class__.__name__)

# Örnek input ile shape testi
# Buradaki depth sayısı örnektir; gerçek batch içinde değişebilir
x = torch.randn(2, 1, 12, 256, 256)  # (B, C, D, H, W)
y = model(x)

print("Input shape :", x.shape)
print("Output shape:", y.shape)  # (B, NUM_CLASSES, D, H, W)

UNet3D
Input shape : torch.Size([2, 1, 12, 256, 256])
Output shape: torch.Size([2, 4, 12, 256, 256])


In [28]:
# =========================================================
# Adım 4C - Checkpoint'i modele yükle
# =========================================================

import os
import torch

# Cihaz seçimi
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Checkpoint yolu
ckpt_path = os.path.join("..", "backend", "models", "best_model.pt")

print("Checkpoint path:", ckpt_path)
print("Dosya var mı?:", os.path.exists(ckpt_path))

# Checkpoint'i yükle
ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

print("\n=== CHECKPOINT BILGISI ===")
print("epoch:", ckpt.get("epoch", None))
print("best_val_dice:", ckpt.get("best_val_dice", None))
print("run_tag:", ckpt.get("run_tag", None))

# Modeli oluştur
model = get_model_3d("unet3d").to(DEVICE)

# Ağırlıkları yükle
model.load_state_dict(ckpt["model_state_dict"])

# Eval moduna al
model.eval()

print("\nModel başarıyla yüklendi.")
print("Device:", DEVICE)
print("Model class:", model.__class__.__name__)

Checkpoint path: ..\backend\models\best_model.pt
Dosya var mı?: True

=== CHECKPOINT BILGISI ===
epoch: 49
best_val_dice: 0.8738915048506688
run_tag: unet3d_gn_wce_dice_seed42

Model başarıyla yüklendi.
Device: cpu
Model class: UNet3D


In [29]:
# =========================================================
# Sürüm A - Adım A1
# processed_tmp klasörünü bul + processed-space spacing mantığı
# =========================================================

from pathlib import Path
import os
import numpy as np

# ---------------------------------------------------------
# 1) processed_tmp klasörünü bul
# ---------------------------------------------------------
# Not:
# preprocess kodunda çıktı klasörü OUT_DIR = processed_tmp idi.
# Bu klasörün içinde:
#   - images/
#   - labels/
# bulunmalı.
#
# Notebook'un konumuna göre birkaç olası yolu kontrol ediyoruz.
# ---------------------------------------------------------
candidate_roots = [
    Path("..") / "processed_tmp",
    Path("processed_tmp"),
    Path("..") / "backend" / "data" / "processed_tmp",
]

PROCESSED_ROOT = None
for p in candidate_roots:
    if (p / "images").exists() and (p / "labels").exists():
        PROCESSED_ROOT = p
        break

print("Bulunan PROCESSED_ROOT:", PROCESSED_ROOT)

if PROCESSED_ROOT is None:
    raise FileNotFoundError(
        "processed_tmp/images ve processed_tmp/labels bulunamadı. "
        "candidate_roots listesini kendi klasör yapına göre güncelle."
    )

IMAGES_DIR = PROCESSED_ROOT / "images"
LABELS_DIR = PROCESSED_ROOT / "labels"

print("IMAGES_DIR:", IMAGES_DIR)
print("LABELS_DIR:", LABELS_DIR)


# ---------------------------------------------------------
# 2) Processed-space spacing oluşturma fonksiyonu
# ---------------------------------------------------------
# Preprocess koduna göre:
# - x ve y spacing sabitleniyor -> (1.37, 1.37) mm
# - z ekseni korunuyor
#
# Bu yüzden processed-space'te klinik hacim hesabı yaparken:
#   spacing_processed = (1.37, 1.37, original_dz)
#
# kullanacağız.
# ---------------------------------------------------------
TARGET_SPACING_XY = (1.37, 1.37)


def build_processed_spacing(original_spacing, target_xy=TARGET_SPACING_XY):
    dx_new = float(target_xy[0])
    dy_new = float(target_xy[1])
    dz_original = float(original_spacing[2])

    spacing_processed = (dx_new, dy_new, dz_original)
    return spacing_processed


# ---------------------------------------------------------
# 3) Bilgi notu
# ---------------------------------------------------------
# Önemli:
# "processed-space" metrikleri ile daha önce NIfTI orijinal uzayında
# hesapladığımız metrikler birebir aynı çıkmak zorunda değildir.
#
# Sebep:
# - x-y spacing değişti
# - veri artık preprocess uzayında
# ---------------------------------------------------------
print("\nNot: Sürüm A'da processed-space klinik parametre hesabı yapacağız.")

Bulunan PROCESSED_ROOT: ..\processed_tmp
IMAGES_DIR: ..\processed_tmp\images
LABELS_DIR: ..\processed_tmp\labels

Not: Sürüm A'da processed-space klinik parametre hesabı yapacağız.


In [30]:
# =========================================================
# Sürüm A - Adım A2
# patient101 için processed image/label dosyalarını bul
# =========================================================

import glob


# ---------------------------------------------------------
# Yardımcı fonksiyon:
# Belirli bir hasta ve frame için processed image/label .npy dosyalarını bul
# ---------------------------------------------------------
def find_processed_pair(
    patient_id, frame, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR
):
    frame_str = f"{int(frame):02d}"

    # preprocess koduna göre isimler şu biçimde kaydediliyor:
    # patient101_frame01.npy
    img_pattern = str(images_dir / f"{patient_id}_frame{frame_str}.npy")
    lbl_pattern = str(labels_dir / f"{patient_id}_frame{frame_str}.npy")

    img_matches = sorted(glob.glob(img_pattern))
    lbl_matches = sorted(glob.glob(lbl_pattern))

    print(f"\n[{patient_id} frame{frame_str}]")
    print("Image pattern:", img_pattern)
    print("Label pattern:", lbl_pattern)
    print("Image matches:", img_matches)
    print("Label matches:", lbl_matches)

    if len(img_matches) == 0:
        raise FileNotFoundError(
            f"Image için eşleşen processed .npy bulunamadı: {img_pattern}"
        )

    if len(lbl_matches) == 0:
        raise FileNotFoundError(
            f"Label için eşleşen processed .npy bulunamadı: {lbl_pattern}"
        )

    return img_matches[0], lbl_matches[0]


# ---------------------------------------------------------
# Şu an patient101 ile gidiyoruz.
# Daha önce GT tarafında:
#   ED = frame01
#   ES = frame14
# kullanmıştık.
# ---------------------------------------------------------
patient_id = "patient001"
ed_frame = 1
es_frame = 12

ed_img_npy, ed_lbl_npy = find_processed_pair(patient_id, ed_frame)
es_img_npy, es_lbl_npy = find_processed_pair(patient_id, es_frame)

print("\n=== SEÇİLEN PROCESSED DOSYALAR ===")
print("ED image npy :", ed_img_npy)
print("ED label npy :", ed_lbl_npy)
print("ES image npy :", es_img_npy)
print("ES label npy :", es_lbl_npy)


[patient001 frame01]
Image pattern: ..\processed_tmp\images\patient001_frame01.npy
Label pattern: ..\processed_tmp\labels\patient001_frame01.npy
Image matches: ['..\\processed_tmp\\images\\patient001_frame01.npy']
Label matches: ['..\\processed_tmp\\labels\\patient001_frame01.npy']

[patient001 frame12]
Image pattern: ..\processed_tmp\images\patient001_frame12.npy
Label pattern: ..\processed_tmp\labels\patient001_frame12.npy
Image matches: ['..\\processed_tmp\\images\\patient001_frame12.npy']
Label matches: ['..\\processed_tmp\\labels\\patient001_frame12.npy']

=== SEÇİLEN PROCESSED DOSYALAR ===
ED image npy : ..\processed_tmp\images\patient001_frame01.npy
ED label npy : ..\processed_tmp\labels\patient001_frame01.npy
ES image npy : ..\processed_tmp\images\patient001_frame12.npy
ES label npy : ..\processed_tmp\labels\patient001_frame12.npy


In [31]:
# =========================================================
# Sürüm A - Adım A3
# processed .npy veriyi modele uygun hale getir
# =========================================================

import numpy as np
import torch


# ---------------------------------------------------------
# Bu fonksiyon eğitim notebook'undaki dataset mantığını takip eder:
# - diskte görüntü ve label (H, W, D) olarak saklı
# - modele vermeden önce (D, H, W)'ye çevriliyor
# - görüntü için batch ve channel boyutu ekleniyor
#
# Çıktılar:
# - x        : model input tensor -> (1, 1, D, H, W)
# - img_dhw  : processed image volume -> (D, H, W)
# - lbl_dhw  : processed label volume -> (D, H, W)
# - info     : debug / kontrol bilgileri
# ---------------------------------------------------------
def load_processed_npy_pair_for_model(img_npy_path, lbl_npy_path, device=DEVICE):
    # Diskten yükle
    img_hwd = np.load(img_npy_path).astype(np.float32)
    lbl_hwd = np.load(lbl_npy_path).astype(np.uint8)

    # Eğitim pipeline mantığı:
    # (H, W, D) -> (D, H, W)
    img_dhw = np.transpose(img_hwd, (2, 0, 1))
    lbl_dhw = np.transpose(lbl_hwd, (2, 0, 1))

    # Tensor'a çevir
    x = torch.from_numpy(np.ascontiguousarray(img_dhw)).float()

    # Model girişi için:
    # (D,H,W) -> (1,D,H,W) -> (1,1,D,H,W)
    x = x.unsqueeze(0).unsqueeze(0).to(device)

    info = {
        "img_path": img_npy_path,
        "lbl_path": lbl_npy_path,
        "img_shape_hwd": img_hwd.shape,
        "lbl_shape_hwd": lbl_hwd.shape,
        "img_shape_dhw": img_dhw.shape,
        "lbl_shape_dhw": lbl_dhw.shape,
        "img_min": float(img_hwd.min()),
        "img_max": float(img_hwd.max()),
        "img_dtype": str(img_hwd.dtype),
        "lbl_unique_labels": np.unique(lbl_hwd).tolist(),
    }

    return x, img_dhw, lbl_dhw, info


# ---------------------------------------------------------
# ED ve ES için processed veriyi yükle
# ---------------------------------------------------------
x_ed, ed_img_dhw, ed_lbl_dhw, ed_proc_info = load_processed_npy_pair_for_model(
    ed_img_npy, ed_lbl_npy, device=DEVICE
)

x_es, es_img_dhw, es_lbl_dhw, es_proc_info = load_processed_npy_pair_for_model(
    es_img_npy, es_lbl_npy, device=DEVICE
)

print("=== ED PROC INFO ===")
for k, v in ed_proc_info.items():
    print(f"{k}: {v}")

print("\n=== ES PROC INFO ===")
for k, v in es_proc_info.items():
    print(f"{k}: {v}")

print("\nED tensor shape:", tuple(x_ed.shape))
print("ES tensor shape:", tuple(x_es.shape))
print("Device:", DEVICE)

=== ED PROC INFO ===
img_path: ..\processed_tmp\images\patient001_frame01.npy
lbl_path: ..\processed_tmp\labels\patient001_frame01.npy
img_shape_hwd: (256, 256, 10)
lbl_shape_hwd: (256, 256, 10)
img_shape_dhw: (10, 256, 256)
lbl_shape_dhw: (10, 256, 256)
img_min: 0.0
img_max: 1.0
img_dtype: float32
lbl_unique_labels: [0, 1, 2, 3]

=== ES PROC INFO ===
img_path: ..\processed_tmp\images\patient001_frame12.npy
lbl_path: ..\processed_tmp\labels\patient001_frame12.npy
img_shape_hwd: (256, 256, 10)
lbl_shape_hwd: (256, 256, 10)
img_shape_dhw: (10, 256, 256)
lbl_shape_dhw: (10, 256, 256)
img_min: 0.0
img_max: 1.0
img_dtype: float32
lbl_unique_labels: [0, 1, 2, 3]

ED tensor shape: (1, 1, 10, 256, 256)
ES tensor shape: (1, 1, 10, 256, 256)
Device: cpu


In [32]:
# =========================================================
# Sürüm A - Adım A4
# processed-space üzerinde 3D model prediction üret
# =========================================================

import numpy as np
import torch


# ---------------------------------------------------------
# Yardımcı fonksiyon:
# Tek bir tensor girişten sınıf maskesi üretir
# ---------------------------------------------------------
@torch.no_grad()
def predict_mask_from_tensor(model, x):
    """
    Girdi:
        x : (1, 1, D, H, W)

    Çıktı:
        pred_dhw : (D, H, W) sınıf etiketi maskesi
        logits   : (1, C, D, H, W)
    """
    model.eval()

    logits = model(x)
    pred_dhw = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

    return pred_dhw, logits


# ---------------------------------------------------------
# ED ve ES prediction üret
# ---------------------------------------------------------
ed_pred_dhw, ed_logits = predict_mask_from_tensor(model, x_ed)
es_pred_dhw, es_logits = predict_mask_from_tensor(model, x_es)

print("=== PREDICTION BILGISI ===")
print("ED pred shape:", ed_pred_dhw.shape)
print("ED pred unique labels:", np.unique(ed_pred_dhw))

print("\nES pred shape:", es_pred_dhw.shape)
print("ES pred unique labels:", np.unique(es_pred_dhw))

=== PREDICTION BILGISI ===
ED pred shape: (10, 256, 256)
ED pred unique labels: [0 1 2 3]

ES pred shape: (10, 256, 256)
ES pred unique labels: [0 1 2 3]


In [49]:
# =========================================================
# Sürüm A - Adım A5
# processed-space klinik parametre hesabı
# =========================================================

# ---------------------------------------------------------
# Burada orijinal NIfTI spacing'ini doğrudan kullanmıyoruz.
# Çünkü preprocess sırasında x-y düzlemi 1.37 mm'ye standardize edildi.
#
# Bu yüzden processed-space spacing:
#   (1.37, 1.37, original_dz)
#
# olacak.
# ---------------------------------------------------------

spacing_processed = build_processed_spacing(ed_raw_spacing, target_xy=TARGET_SPACING_XY)
print("Original spacing (ED):", ed_spacing)
print("Processed spacing    :", spacing_processed)

# ---------------------------------------------------------
# 1) Processed-space GT metrikleri
# ---------------------------------------------------------
metrics_gt_processed = compute_lv_functional_metrics(
    ed_mask=ed_lbl_dhw, es_mask=es_lbl_dhw, spacing=spacing_processed, lv_class_id=3
)

checks_gt_processed, warnings_gt_processed = run_quality_checks(metrics_gt_processed)

print("\n=== PROCESSED-SPACE GT KLINIK PARAMETRELER ===")
for k, v in metrics_gt_processed.items():
    print(f"{k}: {v}")

print("\n=== PROCESSED-SPACE GT KALITE KONTROLLERI ===")
for k, v in checks_gt_processed.items():
    print(f"{k}: {v}")

print("\n=== PROCESSED-SPACE GT UYARILAR ===")
print(warnings_gt_processed if warnings_gt_processed else "Uyarı yok.")


# ---------------------------------------------------------
# 2) Processed-space prediction metrikleri
# ---------------------------------------------------------
metrics_pred_processed = compute_lv_functional_metrics(
    ed_mask=ed_pred_dhw, es_mask=es_pred_dhw, spacing=spacing_processed, lv_class_id=3
)

checks_pred_processed, warnings_pred_processed = run_quality_checks(
    metrics_pred_processed
)

print("\n=== PROCESSED-SPACE PRED KLINIK PARAMETRELER ===")
for k, v in metrics_pred_processed.items():
    print(f"{k}: {v}")

print("\n=== PROCESSED-SPACE PRED KALITE KONTROLLERI ===")
for k, v in checks_pred_processed.items():
    print(f"{k}: {v}")

print("Original spacing used for processed-space conversion:", ed_raw_spacing)
print("Processed spacing    :", spacing_processed)
print("\n=== PROCESSED-SPACE PRED UYARILAR ===")
print(warnings_pred_processed if warnings_pred_processed else "Uyarı yok.")

Original spacing (ED): (np.float32(1.64062), np.float32(1.64062), np.float32(10.0))
Processed spacing    : (1.37, 1.37, 1.0)

=== PROCESSED-SPACE GT KLINIK PARAMETRELER ===
voxel_volume_ml: 0.0018769000000000001
lv_edv_ml: 12.130404700000001
lv_esv_ml: 9.249363200000001
lv_ef_percent: 23.75058022590128

=== PROCESSED-SPACE GT KALITE KONTROLLERI ===
lv_present_in_ed: True
lv_present_in_es: True
edv_greater_than_esv: True
ef_in_valid_range: True

=== PROCESSED-SPACE GT UYARILAR ===
Uyarı yok.

=== PROCESSED-SPACE PRED KLINIK PARAMETRELER ===
voxel_volume_ml: 0.0018769000000000001
lv_edv_ml: 12.1416661
lv_esv_ml: 9.5928359
lv_ef_percent: 20.992425413510585

=== PROCESSED-SPACE PRED KALITE KONTROLLERI ===
lv_present_in_ed: True
lv_present_in_es: True
edv_greater_than_esv: True
ef_in_valid_range: True
Original spacing used for processed-space conversion: (np.float32(1.0), np.float32(1.0), np.float32(1.0))
Processed spacing    : (1.37, 1.37, 1.0)

=== PROCESSED-SPACE PRED UYARILAR ===
Uyarı 

In [46]:
# =========================================================
# Sürüm A - Adım A6
# processed-space GT vs PRED karşılaştırması
# =========================================================


def safe_abs_diff(a, b):
    if a is None or b is None:
        return None
    return abs(a - b)


def safe_signed_diff(pred, gt):
    if pred is None or gt is None:
        return None
    return pred - gt


comparison_processed = {
    "lv_edv_gt_processed_ml": metrics_gt_processed["lv_edv_ml"],
    "lv_edv_pred_processed_ml": metrics_pred_processed["lv_edv_ml"],
    "lv_edv_abs_diff_ml": safe_abs_diff(
        metrics_pred_processed["lv_edv_ml"], metrics_gt_processed["lv_edv_ml"]
    ),
    "lv_edv_signed_diff_ml": safe_signed_diff(
        metrics_pred_processed["lv_edv_ml"], metrics_gt_processed["lv_edv_ml"]
    ),
    "lv_esv_gt_processed_ml": metrics_gt_processed["lv_esv_ml"],
    "lv_esv_pred_processed_ml": metrics_pred_processed["lv_esv_ml"],
    "lv_esv_abs_diff_ml": safe_abs_diff(
        metrics_pred_processed["lv_esv_ml"], metrics_gt_processed["lv_esv_ml"]
    ),
    "lv_esv_signed_diff_ml": safe_signed_diff(
        metrics_pred_processed["lv_esv_ml"], metrics_gt_processed["lv_esv_ml"]
    ),
    "lv_ef_gt_processed_percent": metrics_gt_processed["lv_ef_percent"],
    "lv_ef_pred_processed_percent": metrics_pred_processed["lv_ef_percent"],
    "lv_ef_abs_diff_percent": safe_abs_diff(
        metrics_pred_processed["lv_ef_percent"], metrics_gt_processed["lv_ef_percent"]
    ),
    "lv_ef_signed_diff_percent": safe_signed_diff(
        metrics_pred_processed["lv_ef_percent"], metrics_gt_processed["lv_ef_percent"]
    ),
}

print("=== PROCESSED-SPACE GT vs PRED KARSILASTIRMA ===")
for k, v in comparison_processed.items():
    print(f"{k}: {v}")

=== PROCESSED-SPACE GT vs PRED KARSILASTIRMA ===
lv_edv_gt_processed_ml: 12.130404700000001
lv_edv_pred_processed_ml: 12.1416661
lv_edv_abs_diff_ml: 0.011261399999998645
lv_edv_signed_diff_ml: 0.011261399999998645
lv_esv_gt_processed_ml: 9.249363200000001
lv_esv_pred_processed_ml: 9.5928359
lv_esv_abs_diff_ml: 0.3434726999999995
lv_esv_signed_diff_ml: 0.3434726999999995
lv_ef_gt_processed_percent: 23.75058022590128
lv_ef_pred_processed_percent: 20.992425413510585
lv_ef_abs_diff_percent: 2.7581548123906963
lv_ef_signed_diff_percent: -2.7581548123906963


In [47]:
# =========================================================
# Sürüm A - Adım A7
# kısa özet
# =========================================================

print("=== SÜRÜM A OZET ===")
print(f"GT(processed)   EDV: {metrics_gt_processed['lv_edv_ml']:.2f} mL")
print(f"PRED(processed) EDV: {metrics_pred_processed['lv_edv_ml']:.2f} mL")
print()

print(f"GT(processed)   ESV: {metrics_gt_processed['lv_esv_ml']:.2f} mL")
print(f"PRED(processed) ESV: {metrics_pred_processed['lv_esv_ml']:.2f} mL")
print()

if (
    metrics_gt_processed["lv_ef_percent"] is not None
    and metrics_pred_processed["lv_ef_percent"] is not None
):
    print(f"GT(processed)   EF: {metrics_gt_processed['lv_ef_percent']:.2f} %")
    print(f"PRED(processed) EF: {metrics_pred_processed['lv_ef_percent']:.2f} %")
    print(
        f"EF farkı: "
        f"{abs(metrics_pred_processed['lv_ef_percent'] - metrics_gt_processed['lv_ef_percent']):.2f} %"
    )
else:
    print("Processed-space EF hesaplanamadı.")

=== SÜRÜM A OZET ===
GT(processed)   EDV: 12.13 mL
PRED(processed) EDV: 12.14 mL

GT(processed)   ESV: 9.25 mL
PRED(processed) ESV: 9.59 mL

GT(processed)   EF: 23.75 %
PRED(processed) EF: 20.99 %
EF farkı: 2.76 %


sürüm B

In [36]:
# =========================================================
# Sürüm B - Adım B1
# Raw training verisini bul + ED/ES bilgisi oku + canonical GT yükle
# =========================================================

from pathlib import Path
import os
import re
import nibabel as nib
import numpy as np

# ---------------------------------------------------------
# 1) Raw training root'u bul
# ---------------------------------------------------------
# Bu sürümde processed-space prediction'ı orijinal uzaya bağlayacağımız için
# seçilen hastanın raw/canonical ground-truth maskelerine de ihtiyacımız var.
#
# Olası klasörleri sırayla deniyoruz.
# Senin lokal yapına göre gerekirse candidate_training_roots listesine
# yeni bir yol ekleyebilirsin.
# ---------------------------------------------------------
candidate_training_roots = [
    Path("..") / "database" / "training",
    Path("database") / "training",
    Path("..") / "backend" / "data" / "training",
]

RAW_TRAIN_ROOT = None
for p in candidate_training_roots:
    if p.exists():
        RAW_TRAIN_ROOT = p
        break

print("Bulunan RAW_TRAIN_ROOT:", RAW_TRAIN_ROOT)

if RAW_TRAIN_ROOT is None:
    raise FileNotFoundError(
        "Raw training klasörü bulunamadı. "
        "database/training klasörünü proje yapına göre yerleştir veya "
        "candidate_training_roots listesini güncelle."
    )


# ---------------------------------------------------------
# 2) Info.cfg içinden ED ve ES frame numaralarını okuma
# ---------------------------------------------------------
# ACDC hasta klasörlerinde Info.cfg dosyası bulunur.
# Bu dosyadan hangi frame'in ED, hangisinin ES olduğunu öğreniriz.
# ---------------------------------------------------------
def read_ed_es_from_info_cfg(info_cfg_path):
    with open(info_cfg_path, "r", encoding="utf-8", errors="ignore") as f:
        txt = f.read()

    ed = re.search(r"\bED\s*:\s*(\d+)", txt)
    es = re.search(r"\bES\s*:\s*(\d+)", txt)

    if ed is None or es is None:
        raise ValueError(f"ED/ES bulunamadı: {info_cfg_path}")

    return int(ed.group(1)), int(es.group(1))


# ---------------------------------------------------------
# 3) Canonical orientation ile NIfTI yükle
# ---------------------------------------------------------
# Preprocess kodunda nib.as_closest_canonical kullanılmıştı.
# Bu yüzden original-space ile processed-space arasında daha tutarlı bir köprü
# kurmak için burada da aynı canonical orientation yaklaşımını kullanıyoruz.
# ---------------------------------------------------------
def load_nifti_mask_canonical_as_dhw(nifti_path):
    nii = nib.load(nifti_path)
    nii = nib.as_closest_canonical(nii)

    vol = nii.get_fdata()
    spacing = nii.header.get_zooms()[:3]

    # (H, W, D) -> (D, H, W)
    vol_dhw = np.transpose(vol, (2, 0, 1))
    vol_dhw = np.rint(vol_dhw).astype(np.uint8)

    return vol_dhw, spacing, nii


# ---------------------------------------------------------
# 4) Sürüm B'de Sürüm A ile aynı hastayı kullanıyoruz
# ---------------------------------------------------------
# ÖNEMLİ:
# Sürüm A'da patient001 / frame01 / frame12 ile çalıştık.
# Burada da aynı hasta ve frame'ler ile devam edeceğiz.
# Böylece processed-space ile original-space karşılaştırması tutarlı olur.
# ---------------------------------------------------------
patient_id_b = patient_id
ed_frame_b = ed_frame
es_frame_b = es_frame

patient_dir = RAW_TRAIN_ROOT / patient_id_b
info_cfg_path = patient_dir / "Info.cfg"

print("\nHasta klasörü:", patient_dir)
print("Info.cfg yolu:", info_cfg_path)

if not patient_dir.exists():
    raise FileNotFoundError(f"Hasta klasörü bulunamadı: {patient_dir}")

if not info_cfg_path.exists():
    raise FileNotFoundError(f"Info.cfg bulunamadı: {info_cfg_path}")

ed_frame_cfg, es_frame_cfg = read_ed_es_from_info_cfg(info_cfg_path)

print("\nInfo.cfg -> ED frame:", ed_frame_cfg)
print("Info.cfg -> ES frame:", es_frame_cfg)

# Burada Sürüm A'da kullandığımız frame'lerle Info.cfg'nin tutarlı olup olmadığını kontrol ediyoruz.
print("\nSürüm A'da seçilen ED frame:", ed_frame_b)
print("Sürüm A'da seçilen ES frame:", es_frame_b)

if ed_frame_b != ed_frame_cfg or es_frame_b != es_frame_cfg:
    print("\nUYARI:")
    print("Sürüm A'da kullandığın frame'ler ile Info.cfg'den okunan ED/ES frame'ler birebir aynı değil.")
    print("Bu durumda doğru frame numaralarını Info.cfg'ye göre güncellemen gerekir.")


# ---------------------------------------------------------
# 5) Original-space GT maskelerini yükle
# ---------------------------------------------------------
ed_gt_raw_path = patient_dir / f"{patient_id_b}_frame{ed_frame_cfg:02d}_gt.nii"
es_gt_raw_path = patient_dir / f"{patient_id_b}_frame{es_frame_cfg:02d}_gt.nii"

# Eğer .nii yoksa .nii.gz dene
if not ed_gt_raw_path.exists():
    ed_gt_raw_path = patient_dir / f"{patient_id_b}_frame{ed_frame_cfg:02d}_gt.nii.gz"

if not es_gt_raw_path.exists():
    es_gt_raw_path = patient_dir / f"{patient_id_b}_frame{es_frame_cfg:02d}_gt.nii.gz"

print("\nED GT raw path:", ed_gt_raw_path)
print("ES GT raw path:", es_gt_raw_path)
print("ED GT exists:", ed_gt_raw_path.exists())
print("ES GT exists:", es_gt_raw_path.exists())

ed_gt_raw_dhw, ed_raw_spacing, ed_gt_raw_nii = load_nifti_mask_canonical_as_dhw(ed_gt_raw_path)
es_gt_raw_dhw, es_raw_spacing, es_gt_raw_nii = load_nifti_mask_canonical_as_dhw(es_gt_raw_path)

print("\n=== ORIGINAL-SPACE GT BILGISI ===")
print_mask_info("ED GT RAW", ed_gt_raw_dhw, ed_raw_spacing)
print()
print_mask_info("ES GT RAW", es_gt_raw_dhw, es_raw_spacing)

print("\nOriginal spacing aynı mı?:", np.allclose(ed_raw_spacing, es_raw_spacing))
print("Original shape aynı mı?:", ed_gt_raw_dhw.shape == es_gt_raw_dhw.shape)

Bulunan RAW_TRAIN_ROOT: ..\backend\data\training

Hasta klasörü: ..\backend\data\training\patient001
Info.cfg yolu: ..\backend\data\training\patient001\Info.cfg

Info.cfg -> ED frame: 1
Info.cfg -> ES frame: 12

Sürüm A'da seçilen ED frame: 1
Sürüm A'da seçilen ES frame: 12

ED GT raw path: ..\backend\data\training\patient001\patient001_frame01_gt.nii
ES GT raw path: ..\backend\data\training\patient001\patient001_frame12_gt.nii
ED GT exists: True
ES GT exists: True

=== ORIGINAL-SPACE GT BILGISI ===
ED GT RAW shape (D,H,W): (10, 216, 256)
ED GT RAW spacing (dx,dy,dz): (np.float32(1.0), np.float32(1.0), np.float32(1.0))
ED GT RAW unique labels: [0 1 2 3]

ES GT RAW shape (D,H,W): (10, 216, 256)
ES GT RAW spacing (dx,dy,dz): (np.float32(1.0), np.float32(1.0), np.float32(1.0))
ES GT RAW unique labels: [0 1 2 3]

Original spacing aynı mı?: True
Original shape aynı mı?: True


In [39]:
# =========================================================
# Sürüm B - Adım B2
# processed-space maskeyi original-space'e geri taşıma yardımcı fonksiyonları
# ========================================================

import cv2
import numpy as np

# ---------------------------------------------------------
# 1) Preprocess'teki resampled H/W boyutunu yeniden hesapla
# ---------------------------------------------------------
# Preprocess kodunda şu formül kullanılıyordu:
#
# H_new = round(H * (sx / tx))
# W_new = round(W * (sy / ty))
#
# Burada:
# - (H, W) = canonical raw görüntü boyutu
# - sx, sy = original spacing
# - tx, ty = hedef spacing = (1.37, 1.37)
#
# Bu fonksiyon, original görüntüden processed-space öncesi resampled boyutu üretir.
# ---------------------------------------------------------
def compute_resampled_hw_from_original(original_hw, original_spacing_xy, target_xy=(1.37, 1.37)):
    H, W = original_hw
    sx, sy = float(original_spacing_xy[0]), float(original_spacing_xy[1])
    tx, ty = float(target_xy[0]), float(target_xy[1])

    H_new = max(1, int(round(H * (sx / tx))))
    W_new = max(1, int(round(W * (sy / ty))))

    return H_new, W_new


# ---------------------------------------------------------
# 2) crop/pad işlemini yaklaşık olarak tersle
# ---------------------------------------------------------
# Preprocess sırasında 256x256 çıktı elde etmek için center crop / pad uygulanmıştı.
#
# Bu tersleme tam olarak bilgi geri getirmez:
# - eğer preprocess sırasında crop olduysa, kırpılan dış bölgeler kaybolmuştur
# - burada onları sıfır kabul ederek merkezi bölgeyi geri yerleştiriyoruz
#
# Yani bu fonksiyon "yaklaşık inverse" yapar.
# ---------------------------------------------------------
def invert_center_crop_pad_2d(mask_256, original_resampled_hw, target_hw=(256, 256)):
    H_res, W_res = original_resampled_hw
    Ht, Wt = target_hw

    recovered = np.zeros((H_res, W_res), dtype=np.uint8)

    # Preprocess'teki crop_or_pad_to_size mantığının aynı indeksleri:
    y0 = max(0, (Ht - H_res) // 2)
    x0 = max(0, (Wt - W_res) // 2)

    ys = max(0, (H_res - Ht) // 2)
    xs = max(0, (W_res - Wt) // 2)

    ye = ys + min(H_res, Ht)
    xe = xs + min(W_res, Wt)
    y1 = y0 + (ye - ys)
    x1 = x0 + (xe - xs)

    # Processed 256x256 maskenin ilgili merkez bölgesini
    # tekrar resampled boyutlu alana yerleştiriyoruz.
    recovered[ys:ye, xs:xe] = mask_256[y0:y1, x0:x1].astype(np.uint8)

    return recovered


# ---------------------------------------------------------
# 3) In-plane resample'i tersle
# ---------------------------------------------------------
# Processed-space'teki maske önce resampled uzaya, sonra original H/W'ye
# nearest-neighbor ile geri taşınır.
#
# Neden nearest-neighbor?
# Çünkü label maskelerinde ara sınıf üretmek istemeyiz.
# ---------------------------------------------------------
def resize_mask_to_original_hw(mask_hw, original_hw):
    H_orig, W_orig = original_hw

    out = cv2.resize(
        mask_hw.astype(np.uint8),
        (W_orig, H_orig),
        interpolation=cv2.INTER_NEAREST
    )

    return out.astype(np.uint8)


# ---------------------------------------------------------
# 4) Tüm hacim için processed-space -> original-space dönüşümü
# ---------------------------------------------------------
# Girdi:
#   pred_dhw              : (D, 256, 256)
#   original_dhw_shape    : (D, H_orig, W_orig)
#   original_spacing      : (dx, dy, dz)
#
# Adımlar:
#   a) original H/W ve spacing'e göre preprocess öncesi resampled H/W bulunur
#   b) her slice için 256x256 maske -> resampled H/W uzayı
#   c) oradan original H/W'ye nearest-neighbor ile geri dönülür
#   d) tüm slice'lar birleştirilir
# ---------------------------------------------------------
def invert_processed_prediction_to_original_space(
    pred_dhw,
    original_dhw_shape,
    original_spacing,
    target_xy=(1.37, 1.37),
    target_hw=(256, 256)
):
    D_orig, H_orig, W_orig = original_dhw_shape

    # Z ekseni preprocess'te korunuyordu; yine de burada bir güvenlik kontrolü yapıyoruz.
    if pred_dhw.shape[0] != D_orig:
        raise ValueError(
            f"Depth uyuşmuyor. pred depth={pred_dhw.shape[0]}, original depth={D_orig}"
        )

    # Preprocess'te kullanılan resampled H/W boyutunu yeniden hesapla
    H_res, W_res = compute_resampled_hw_from_original(
        original_hw=(H_orig, W_orig),
        original_spacing_xy=(original_spacing[0], original_spacing[1]),
        target_xy=target_xy
    )

    pred_original_dhw = np.zeros((D_orig, H_orig, W_orig), dtype=np.uint8)

    for d in range(D_orig):
        mask_256 = pred_dhw[d]  # (256, 256)

        # 1) 256x256 -> preprocess sonrası resampled uzay
        mask_res_hw = invert_center_crop_pad_2d(
            mask_256,
            original_resampled_hw=(H_res, W_res),
            target_hw=target_hw
        )

        # 2) resampled uzay -> original H/W
        mask_orig_hw = resize_mask_to_original_hw(
            mask_res_hw,
            original_hw=(H_orig, W_orig)
        )

        pred_original_dhw[d] = mask_orig_hw

    meta = {
        "original_shape_dhw": original_dhw_shape,
        "original_spacing": tuple(float(x) for x in original_spacing),
        "resampled_hw_before_crop_pad": (H_res, W_res),
        "target_hw": target_hw,
        "target_xy": target_xy,
    }

    return pred_original_dhw, meta

In [40]:
# =========================================================
# Sürüm B - Adım B3
# processed-space prediction maskelerini original-space'e taşı
# =========================================================

# ---------------------------------------------------------
# Burada Sürüm A'da ürettiğimiz prediction'ları kullanıyoruz:
#   ed_pred_dhw
#   es_pred_dhw
#
# Bunlar processed-space'te idi.
# Şimdi bunları patient001'in original canonical uzayına geri taşıyoruz.
# ---------------------------------------------------------
ed_pred_original_dhw, ed_inverse_meta = invert_processed_prediction_to_original_space(
    pred_dhw=ed_pred_dhw,
    original_dhw_shape=ed_gt_raw_dhw.shape,
    original_spacing=ed_raw_spacing,
    target_xy=TARGET_SPACING_XY,
    target_hw=(256, 256)
)

es_pred_original_dhw, es_inverse_meta = invert_processed_prediction_to_original_space(
    pred_dhw=es_pred_dhw,
    original_dhw_shape=es_gt_raw_dhw.shape,
    original_spacing=es_raw_spacing,
    target_xy=TARGET_SPACING_XY,
    target_hw=(256, 256)
)

print("=== ED ORIGINAL-SPACE PRED BILGISI ===")
print("ED pred original shape:", ed_pred_original_dhw.shape)
print("ED pred original unique labels:", np.unique(ed_pred_original_dhw))
print("ED inverse meta:", ed_inverse_meta)

print("\n=== ES ORIGINAL-SPACE PRED BILGISI ===")
print("ES pred original shape:", es_pred_original_dhw.shape)
print("ES pred original unique labels:", np.unique(es_pred_original_dhw))
print("ES inverse meta:", es_inverse_meta)

=== ED ORIGINAL-SPACE PRED BILGISI ===
ED pred original shape: (10, 216, 256)
ED pred original unique labels: [0 1 2 3]
ED inverse meta: {'original_shape_dhw': (10, 216, 256), 'original_spacing': (1.0, 1.0, 1.0), 'resampled_hw_before_crop_pad': (158, 187), 'target_hw': (256, 256), 'target_xy': (1.37, 1.37)}

=== ES ORIGINAL-SPACE PRED BILGISI ===
ES pred original shape: (10, 216, 256)
ES pred original unique labels: [0 1 2 3]
ES inverse meta: {'original_shape_dhw': (10, 216, 256), 'original_spacing': (1.0, 1.0, 1.0), 'resampled_hw_before_crop_pad': (158, 187), 'target_hw': (256, 256), 'target_xy': (1.37, 1.37)}


In [41]:
# =========================================================
# Sürüm B - Adım B4
# original-space klinik parametre hesabı
# =========================================================

# ---------------------------------------------------------
# Bu aşamada artık original canonical GT maskeleri ile
# original uzaya geri taşınmış prediction maskelerini karşılaştırıyoruz.
#
# Spacing olarak doğrudan original NIfTI spacing'ini kullanıyoruz.
# ---------------------------------------------------------
spacing_original = ed_raw_spacing

print("Original spacing used for clinical metrics:", spacing_original)

# ---------------------------------------------------------
# 1) Original-space GT metrikleri
# ---------------------------------------------------------
metrics_gt_original = compute_lv_functional_metrics(
    ed_mask=ed_gt_raw_dhw,
    es_mask=es_gt_raw_dhw,
    spacing=spacing_original,
    lv_class_id=3
)

checks_gt_original, warnings_gt_original = run_quality_checks(metrics_gt_original)

print("\n=== ORIGINAL-SPACE GT KLINIK PARAMETRELER ===")
for k, v in metrics_gt_original.items():
    print(f"{k}: {v}")

print("\n=== ORIGINAL-SPACE GT KALITE KONTROLLERI ===")
for k, v in checks_gt_original.items():
    print(f"{k}: {v}")

print("\n=== ORIGINAL-SPACE GT UYARILAR ===")
print(warnings_gt_original if warnings_gt_original else "Uyarı yok.")


# ---------------------------------------------------------
# 2) Original-space prediction metrikleri
# ---------------------------------------------------------
metrics_pred_original = compute_lv_functional_metrics(
    ed_mask=ed_pred_original_dhw,
    es_mask=es_pred_original_dhw,
    spacing=spacing_original,
    lv_class_id=3
)

checks_pred_original, warnings_pred_original = run_quality_checks(metrics_pred_original)

print("\n=== ORIGINAL-SPACE PRED KLINIK PARAMETRELER ===")
for k, v in metrics_pred_original.items():
    print(f"{k}: {v}")

print("\n=== ORIGINAL-SPACE PRED KALITE KONTROLLERI ===")
for k, v in checks_pred_original.items():
    print(f"{k}: {v}")

print("\n=== ORIGINAL-SPACE PRED UYARILAR ===")
print(warnings_pred_original if warnings_pred_original else "Uyarı yok.")

Original spacing used for clinical metrics: (np.float32(1.0), np.float32(1.0), np.float32(1.0))

=== ORIGINAL-SPACE GT KLINIK PARAMETRELER ===
voxel_volume_ml: 0.0010000000474974513
lv_edv_ml: 12.10400104522705
lv_esv_ml: 9.241000175476074
lv_ef_percent: 23.653343200683594

=== ORIGINAL-SPACE GT KALITE KONTROLLERI ===
lv_present_in_ed: True
lv_present_in_es: True
edv_greater_than_esv: True
ef_in_valid_range: True

=== ORIGINAL-SPACE GT UYARILAR ===
Uyarı yok.

=== ORIGINAL-SPACE PRED KLINIK PARAMETRELER ===
voxel_volume_ml: 0.0010000000474974513
lv_edv_ml: 12.133000373840332
lv_esv_ml: 9.568000793457031
lv_ef_percent: 21.140687942504883

=== ORIGINAL-SPACE PRED KALITE KONTROLLERI ===
lv_present_in_ed: True
lv_present_in_es: True
edv_greater_than_esv: True
ef_in_valid_range: True

=== ORIGINAL-SPACE PRED UYARILAR ===
Uyarı yok.


In [43]:
# =========================================================
# Sürüm B - Adım B5
# original-space GT vs PRED karşılaştırması
# =========================================================

comparison_original = {
    "lv_edv_gt_original_ml": metrics_gt_original["lv_edv_ml"],
    "lv_edv_pred_original_ml": metrics_pred_original["lv_edv_ml"],
    "lv_edv_abs_diff_ml": safe_abs_diff(
        metrics_pred_original["lv_edv_ml"],
        metrics_gt_original["lv_edv_ml"]
    ),
    "lv_edv_signed_diff_ml": safe_signed_diff(
        metrics_pred_original["lv_edv_ml"],
        metrics_gt_original["lv_edv_ml"]
    ),

    "lv_esv_gt_original_ml": metrics_gt_original["lv_esv_ml"],
    "lv_esv_pred_original_ml": metrics_pred_original["lv_esv_ml"],
    "lv_esv_abs_diff_ml": safe_abs_diff(
        metrics_pred_original["lv_esv_ml"],
        metrics_gt_original["lv_esv_ml"]
    ),
    "lv_esv_signed_diff_ml": safe_signed_diff(
        metrics_pred_original["lv_esv_ml"],
        metrics_gt_original["lv_esv_ml"]
    ),

    "lv_ef_gt_original_percent": metrics_gt_original["lv_ef_percent"],
    "lv_ef_pred_original_percent": metrics_pred_original["lv_ef_percent"],
    "lv_ef_abs_diff_percent": safe_abs_diff(
        metrics_pred_original["lv_ef_percent"],
        metrics_gt_original["lv_ef_percent"]
    ),
    "lv_ef_signed_diff_percent": safe_signed_diff(
        metrics_pred_original["lv_ef_percent"],
        metrics_gt_original["lv_ef_percent"]
    ),
}

print("=== ORIGINAL-SPACE GT vs PRED KARSILASTIRMA ===")
for k, v in comparison_original.items():
    print(f"{k}: {v}")

=== ORIGINAL-SPACE GT vs PRED KARSILASTIRMA ===
lv_edv_gt_original_ml: 12.10400104522705
lv_edv_pred_original_ml: 12.133000373840332
lv_edv_abs_diff_ml: 0.02899932861328125
lv_edv_signed_diff_ml: 0.02899932861328125
lv_esv_gt_original_ml: 9.241000175476074
lv_esv_pred_original_ml: 9.568000793457031
lv_esv_abs_diff_ml: 0.32700061798095703
lv_esv_signed_diff_ml: 0.32700061798095703
lv_ef_gt_original_percent: 23.653343200683594
lv_ef_pred_original_percent: 21.140687942504883
lv_ef_abs_diff_percent: 2.512655258178711
lv_ef_signed_diff_percent: -2.512655258178711


In [44]:
# =========================================================
# Sürüm B - Adım B6
# kısa özet + metodolojik not
# =========================================================

print("=== SÜRÜM B OZET ===")
print(f"GT(original)   EDV: {metrics_gt_original['lv_edv_ml']:.2f} mL")
print(f"PRED(original) EDV: {metrics_pred_original['lv_edv_ml']:.2f} mL")
print()

print(f"GT(original)   ESV: {metrics_gt_original['lv_esv_ml']:.2f} mL")
print(f"PRED(original) ESV: {metrics_pred_original['lv_esv_ml']:.2f} mL")
print()

if (
    metrics_gt_original["lv_ef_percent"] is not None and
    metrics_pred_original["lv_ef_percent"] is not None
):
    print(f"GT(original)   EF: {metrics_gt_original['lv_ef_percent']:.2f} %")
    print(f"PRED(original) EF: {metrics_pred_original['lv_ef_percent']:.2f} %")
    print(
        f"EF farkı: "
        f"{abs(metrics_pred_original['lv_ef_percent'] - metrics_gt_original['lv_ef_percent']):.2f} %"
    )
else:
    print("Original-space EF hesaplanamadı.")

print("\n=== METODOLOJIK NOT ===")
print(
    "Bu sürümde prediction maskesi processed-space'ten original-space'e yaklaşık olarak geri taşınmıştır. "
    "Preprocess sırasında crop uygulanmışsa, kırpılan dış bölgeler geri getirilemez; "
    "bu nedenle bu sürüm klinik olarak daha anlamlı olsa da tam kayıpsız bir inverse değildir."
)

=== SÜRÜM B OZET ===
GT(original)   EDV: 12.10 mL
PRED(original) EDV: 12.13 mL

GT(original)   ESV: 9.24 mL
PRED(original) ESV: 9.57 mL

GT(original)   EF: 23.65 %
PRED(original) EF: 21.14 %
EF farkı: 2.51 %

=== METODOLOJIK NOT ===
Bu sürümde prediction maskesi processed-space'ten original-space'e yaklaşık olarak geri taşınmıştır. Preprocess sırasında crop uygulanmışsa, kırpılan dış bölgeler geri getirilemez; bu nedenle bu sürüm klinik olarak daha anlamlı olsa da tam kayıpsız bir inverse değildir.
